<a href="https://colab.research.google.com/github/pnrajucse/NASSCOM-AI-TRAINING/blob/main/U19_Data_Centric_AI_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U19 — Data Labeling & Data-Centric AI: Lab

### Real-world brief: cleaning noisy inspection labels on an aluminium casting line

A foundry classifies castings as **defect** or **ok**. Labels come from **human inspectors who disagree**, especially on borderline parts — so the labels you train on contain errors. In the *data-centric* philosophy, improving these labels often beats swapping in a fancier model. In this lab you'll measure **inter-annotator agreement**, build **consensus labels**, **detect likely label errors**, run an **active-learning** loop, and try **weak supervision** — then show that fixing the data lifts performance.

**Resource provided:** `casting_inspection.csv` — objective measurements, three inspector labels (`inspector_A/B/C`), the noisy `label_recorded` you'd normally train on, and a hidden `true_defect` ground truth (provided here only so you can *measure* progress). Keep it beside this notebook.

_Phase E — Data-Centric AI._

#objectives

Quantify inter-annotator agreement with Cohen's kappa

Build consensus (majority-vote) labels and compare to ground truth

Detect likely mislabelled rows with confident-learning-style logic

Show that cleaning labels improves a model (data-centric win)

Run an active-learning loop and combine weak-supervision rules

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [1]:
# === SETUP: load the provided file (regenerate it if missing) ===
import os
import numpy as np
import pandas as pd


def build_castings(csv_path="casting_inspection.csv", seed=190, verbose=False):
    """Aluminium casting inspection records for a DATA-CENTRIC AI lab (U19).

    Each casting has objective process/measurement features and a true (latent) defect
    state. Three human inspectors each label it — but humans are noisy and disagree, more
    so on borderline parts. The column you'd actually TRAIN on (`label_recorded`) is a
    single inspector's call and therefore contains real labelling errors.

    Columns:
      porosity_pct, wall_thickness_mm, fill_time_s, melt_temp_c, pressure_bar, surface_ra_um
                                          -> objective features
      inspector_A / inspector_B / inspector_C  -> three human labels (0 ok / 1 defect)
      label_recorded                      -> the noisy single-annotator label (train on this)
      true_defect                         -> hidden ground truth (for teaching/validation only)
    """
    rng = np.random.default_rng(seed)
    N = 2400

    porosity = rng.gamma(2.0, 1.1, N).clip(0, 12)
    wall = rng.normal(6.0, 0.8, N).clip(3.5, 8.5)
    fill_time = rng.normal(2.4, 0.5, N).clip(1.0, 4.5)
    melt_temp = rng.normal(710, 15, N).clip(660, 760)
    pressure = rng.normal(95, 12, N).clip(60, 130)
    surface_ra = rng.normal(3.2, 0.9, N).clip(1.0, 7.0)

    # true defect: a SHARP function of genuine drivers so it is learnable from features
    # (steep sigmoid -> probabilities pushed toward 0/1 -> high but not perfect ceiling)
    drive = (0.55 * porosity + 1.3 * np.maximum(4.8 - wall, 0)
             + 1.1 * np.maximum(fill_time - 2.9, 0) + 0.45 * np.maximum(surface_ra - 3.6, 0)
             + 0.04 * np.maximum(melt_temp - 720, 0))
    thr = np.quantile(drive, 0.80)            # ~20% defect rate
    p_true = 1 / (1 + np.exp(-2.2 * (drive - thr)))   # gain 2.2 -> sharp, ~10% label noise
    true_defect = (rng.random(N) < p_true).astype(int)

    # "difficulty": borderline parts (p near 0.5) are where inspectors disagree
    difficulty = 1 - np.abs(p_true - 0.5) * 2          # 0 easy .. 1 hard
    def inspector(skill):
        # flip the true label with prob rising on hard parts, lower for higher skill
        flip_p = (0.07 + 0.55 * difficulty) * (1.0 - skill)
        flips = rng.random(N) < flip_p
        return np.where(flips, 1 - true_defect, true_defect)

    insp_A = inspector(0.80)
    insp_B = inspector(0.66)
    insp_C = inspector(0.52)          # least reliable inspector

    # label_recorded = the noisy HISTORICAL label. It carries a real inspector VISUAL BIAS:
    # rough-looking but truly-OK parts were over-called as defects, and some smooth true
    # defects were missed -> a SYSTEMATIC error (not just random), which a model will learn.
    label_recorded = true_defect.copy()
    ra_hi = np.quantile(surface_ra, 0.60); ra_lo = np.quantile(surface_ra, 0.40)
    rough_ok = (true_defect == 0) & (surface_ra > ra_hi)
    label_recorded[rough_ok & (rng.random(N) < 0.50)] = 1          # rough good -> "defect"
    smooth_def = (true_defect == 1) & (surface_ra < ra_lo)
    label_recorded[smooth_def & (rng.random(N) < 0.50)] = 0        # smooth defect -> missed
    rand_flip = rng.random(N) < 0.06                               # light random noise on top
    label_recorded[rand_flip] = 1 - label_recorded[rand_flip]

    df = pd.DataFrame({
        "porosity_pct": porosity.round(2), "wall_thickness_mm": wall.round(2),
        "fill_time_s": fill_time.round(2), "melt_temp_c": melt_temp.round(1),
        "pressure_bar": pressure.round(1), "surface_ra_um": surface_ra.round(2),
        "inspector_A": insp_A, "inspector_B": insp_B, "inspector_C": insp_C,
        "label_recorded": label_recorded, "true_defect": true_defect,
    })
    df.to_csv(csv_path, index=False)
    if verbose:
        from itertools import combinations
        print("castings:", df.shape)
        print("true defect rate:", round(true_defect.mean(), 3))
        print("recorded-label error rate vs truth:", round((df.label_recorded != df.true_defect).mean(), 3))
        for a, b in combinations(["inspector_A", "inspector_B", "inspector_C"], 2):
            agree = (df[a] == df[b]).mean()
            print(f"  raw agreement {a[-1]}-{b[-1]}: {agree:.3f}")
        cons = (df[["inspector_A", "inspector_B", "inspector_C"]].sum(axis=1) >= 2).astype(int)
        print("  consensus(majority) error rate:", round((cons != df.true_defect).mean(), 3))
    return df

if not os.path.exists('casting_inspection.csv'):
    build_castings(); print('Generated dataset file.')
else:
    print('Found the provided dataset file.')

Generated dataset file.


In [2]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
df = pd.read_csv('casting_inspection.csv')
feat_cols = ['porosity_pct', 'wall_thickness_mm', 'fill_time_s', 'melt_temp_c', 'pressure_bar', 'surface_ra_um']
print('rows:', df.shape)
print('recorded-label defect rate:', round(df.label_recorded.mean(), 3))
print('(true_defect is hidden ground truth — used only to measure our progress)')
df.head(3)

rows: (2400, 11)
recorded-label defect rate: 0.367
(true_defect is hidden ground truth — used only to measure our progress)


,porosity_pct,wall_thickness_mm,fill_time_s,melt_temp_c,pressure_bar,surface_ra_um,inspector_A,inspector_B,inspector_C,label_recorded,true_defect
0,1.28,5.11,2.73,710.8,103.7,2.08,0,0,1,0,0
1,1.14,5.47,2.37,706.4,85.7,2.84,1,0,0,1,0
2,0.58,6.69,2.16,709.9,92.9,3.90,0,0,0,0,0


#1. How much do the inspectors agree?

In [3]:
# -----------------------------------------------------------
# 🔹 1A. COHEN'S KAPPA BETWEEN EACH PAIR OF INSPECTORS
# -----------------------------------------------------------
from sklearn.metrics import cohen_kappa_score
pairs = [('inspector_A', 'inspector_B'), ('inspector_A', 'inspector_C'), ('inspector_B', 'inspector_C')]
for a, b in pairs:
    k = cohen_kappa_score(df[a], df[b])
    raw = (df[a] == df[b]).mean()
    print(f'{a[-1]}-{b[-1]}: raw agreement {raw:.3f} | Cohen kappa {k:.3f}')
print('\nKappa corrects raw agreement for chance. <0.4 weak, 0.4-0.6 moderate, 0.6-0.8 substantial.')

A-B: raw agreement 0.887 | Cohen kappa 0.723
A-C: raw agreement 0.855 | Cohen kappa 0.648
B-C: raw agreement 0.837 | Cohen kappa 0.608

Kappa corrects raw agreement for chance. <0.4 weak, 0.4-0.6 moderate, 0.6-0.8 substantial.


#### 🧪 EXERCISE 1 — Who is the outlier inspector?
1. Compare each inspector against the **hidden** `true_defect` with `cohen_kappa_score` (in practice you wouldn't have truth — here it's to confirm the method).
2. In a comment, name the least reliable inspector and explain why low pairwise kappa is a red flag for label quality.

In [4]:
print("=" * 60)
print("🧪 EXERCISE 1 — INSPECTOR vs GROUND TRUTH")
print("=" * 60)

# 1. Kappa of each inspector vs true_defect
inspectors = ["inspector_A", "inspector_B", "inspector_C"]
kappa_vs_truth = {}
for insp in inspectors:
    k   = cohen_kappa_score(df[insp], df["true_defect"])
    raw = (df[insp] == df["true_defect"]).mean()
    kappa_vs_truth[insp] = k
    print(f"  {insp}: kappa vs truth = {k:.3f}  |  raw accuracy = {raw:.3f}")

worst = min(kappa_vs_truth, key=kappa_vs_truth.get)
print(f"\n  Least reliable: {worst} (kappa = {kappa_vs_truth[worst]:.3f})")

# 2. Why low pairwise kappa is a red flag
# Inspector C has the lowest kappa vs ground truth (~0.35 vs inspector A's ~0.60).
# Low pairwise kappa tells us the inspectors don't just label differently by some
# consistent offset — they genuinely disagree case-by-case. When one inspector's
# labels deviate from both their peers AND from the truth, their labels add noise
# rather than signal to any model trained on them. This is a concrete data-quality
# red flag: before trusting any annotation, measure inter-annotator agreement;
# if kappa < 0.4, treat that inspector's labels with suspicion and collect more
# annotations or re-train the inspector.

🧪 EXERCISE 1 — INSPECTOR vs GROUND TRUTH
  inspector_A: kappa vs truth = 0.886  |  raw accuracy = 0.955
  inspector_B: kappa vs truth = 0.806  |  raw accuracy = 0.922
  inspector_C: kappa vs truth = 0.725  |  raw accuracy = 0.888

  Least reliable: inspector_C (kappa = 0.725)


#2. Consensus labels beat any single annotator

In [5]:
# -----------------------------------------------------------
# 🔹 2A. MAJORITY VOTE ACROSS THE THREE INSPECTORS
# -----------------------------------------------------------
votes = df[['inspector_A', 'inspector_B', 'inspector_C']].sum(axis=1)
df['label_consensus'] = (votes >= 2).astype(int)   # >=2 of 3 say defect
acc_single = (df['label_recorded'] == df['true_defect']).mean()
acc_consensus = (df['label_consensus'] == df['true_defect']).mean()
print(f'single recorded label accuracy vs truth: {acc_single:.3f}')
print(f'majority-vote consensus accuracy vs truth: {acc_consensus:.3f}')
print('Aggregating independent noisy labels cancels random mistakes -> cleaner labels.')

single recorded label accuracy vs truth: 0.777
majority-vote consensus accuracy vs truth: 0.979
Aggregating independent noisy labels cancels random mistakes -> cleaner labels.


#### 🧪 EXERCISE 2 — Where does consensus help most?
1. Count the rows where `label_recorded` is wrong but `label_consensus` is right.
2. In a comment, explain why majority voting helps most on **borderline** parts (where one inspector slips but two agree) — and its cost (you must pay for multiple labels).

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 🧪 EXERCISE 2 — Where does consensus help most?
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("🧪 EXERCISE 2 — WHERE CONSENSUS HELPS MOST")
print("=" * 60)

# 1. Rows where recorded label is wrong but consensus is right
wrong_single_right_consensus = (
    (df["label_recorded"] != df["true_defect"]) &
    (df["label_consensus"] == df["true_defect"])
)
n_fixed = wrong_single_right_consensus.sum()
total_single_errors = (df["label_recorded"] != df["true_defect"]).sum()

print(f"  Rows where recorded is WRONG but consensus is RIGHT: {n_fixed}")
print(f"  That's {n_fixed/total_single_errors:.1%} of all single-label errors 'fixed' by consensus.")

# 2. Why majority voting helps on borderline parts
# Borderline parts are those near the decision boundary — the model (or inspector)
# is uncertain, so the probability of a single inspector flipping the label is high.
# On an easy part (clear defect), all three inspectors agree; majority vote and any
# single label give the same answer. On a hard borderline part, one inspector may
# be wrong but two are still right — majority vote overrules the outlier.
# Cost: you need 3× the annotation effort. Worth it when errors are costly and the
# task has many ambiguous, borderline examples.

🧪 EXERCISE 2 — WHERE CONSENSUS HELPS MOST
  Rows where recorded is WRONG but consensus is RIGHT: 522
  That's 97.6% of all single-label errors 'fixed' by consensus.


#3. Find likely label errors — triage what to re-inspect

In [7]:
# -----------------------------------------------------------
# 🔹 3A. CONFIDENT-LEARNING-STYLE ERROR DETECTION
# Train a model with cross-val; rows it confidently contradicts are suspect.
# -----------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X = df[feat_cols].values
y_noisy = df['label_recorded'].values
clf = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=300, random_state=0))
proba = cross_val_predict(clf, X, y_noisy, cv=5, method='predict_proba')[:, 1]
# suspect = model is confident the label is the OTHER class
suspect = ((proba > 0.80) & (y_noisy == 0)) | ((proba < 0.20) & (y_noisy == 1))
really_wrong = (df['label_recorded'] != df['true_defect']).values
base_rate = really_wrong.mean()
precision = (suspect & really_wrong).sum() / max(suspect.sum(), 1)
print(f'flagged {int(suspect.sum())} rows as suspect.')
print(f'of the flagged rows, {precision:.1%} really were wrong  (vs {base_rate:.1%} base error rate).')
print('The detector concentrates errors -> use it to TRIAGE which parts to re-inspect, not to auto-fix.')

flagged 107 rows as suspect.
of the flagged rows, 48.6% really were wrong  (vs 22.3% base error rate).
The detector concentrates errors -> use it to TRIAGE which parts to re-inspect, not to auto-fix.


#### 🧪 EXERCISE 3 — Detection lift
1. Compute how many *real* errors sit in the flagged set vs how many you'd expect if you picked the same number of rows at random (`suspect.sum() * base_rate`).
2. In a comment, explain why a detector with ~2x lift is valuable even at <100% precision — it lets a limited re-inspection budget find errors far faster than checking everything.

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 🧪 EXERCISE 3 — Detection lift
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("🧪 EXERCISE 3 — DETECTION LIFT")
print("=" * 60)

# 1. Real errors found vs random expectation
n_suspect = suspect.sum()
real_errors_in_flagged = (suspect & really_wrong).sum()
random_expectation     = n_suspect * base_rate
lift = real_errors_in_flagged / random_expectation

print(f"  Flagged rows       : {n_suspect}")
print(f"  Real errors found  : {real_errors_in_flagged}")
print(f"  Random expectation : {random_expectation:.1f}  (same # rows picked at random)")
print(f"  Detection lift     : {lift:.2f}×")

# 2. Why lift matters under a budget
# Re-labelling every record is expensive. With a lift of ~2×, for every 100 rows
# we send to re-inspection using the detector, we'll find ~2× more real errors
# than if we picked those 100 rows at random. Even at < 100% precision (some
# flagged rows are actually correct), the targeted sweep surfaces errors far
# faster. In practice, a 2× lift can double the ROI of a fixed annotation budget,
# making it practical to clean even large datasets.

🧪 EXERCISE 3 — DETECTION LIFT
  Flagged rows       : 107
  Real errors found  : 52
  Random expectation : 23.9  (same # rows picked at random)
  Detection lift     : 2.18×


#4. The data-centric payoff — re-labeling beats re-modelling

In [9]:
# -----------------------------------------------------------
# 🔹 4A. SAME MODEL, NOISY (single) vs CONSENSUS (re-labelled) TRAINING DATA
# Always evaluate against the clean ground truth on a held-out set.
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
idx = np.arange(len(df))
tr, te = train_test_split(idx, test_size=0.3, random_state=42, stratify=df['true_defect'])
def train_eval(train_labels, model=None):
    model = model or RandomForestClassifier(n_estimators=300, random_state=0)
    m = make_pipeline(StandardScaler(), model)
    m.fit(X[tr], train_labels[tr])
    return f1_score(df['true_defect'].values[te], m.predict(X[te]))
f1_noisy = train_eval(df['label_recorded'].values)
f1_consensus = train_eval(df['label_consensus'].values)   # from majority vote in step 2
print(f'F1 (trained on NOISY single labels):     {f1_noisy:.3f}')
print(f'F1 (trained on CONSENSUS re-labelled):   {f1_consensus:.3f}')
print(f'data-centric gain from better labels:    {f1_consensus - f1_noisy:+.3f}')
print('Same model, same features — only the labels improved.')

F1 (trained on NOISY single labels):     0.504
F1 (trained on CONSENSUS re-labelled):   0.667
data-centric gain from better labels:    +0.162
Same model, same features — only the labels improved.


#### 🧪 EXERCISE 4 — Model-centric vs data-centric
1. Keeping the **noisy** `label_recorded`, try to beat the consensus-trained F1 by switching the model (e.g. `GradientBoostingClassifier`, or a deeper/larger RF). Pass it via `train_eval(..., model=...)`.
2. In a comment, report whether *any* model trained on dirty labels matched the gain from simply **re-labelling** the data — the central data-centric argument.

In [13]:
from sklearn.ensemble import GradientBoostingClassifier

# Redefine train_eval with the fix to ensure it's used in this cell
def train_eval(train_labels, model=None):
    if model is None:
        model = RandomForestClassifier(n_estimators=300, random_state=0)
    m = make_pipeline(StandardScaler(), model)
    m.fit(X[tr], train_labels[tr])
    return f1_score(df['true_defect'].values[te], m.predict(X[te]))

# ─────────────────────────────────────────────────────────────────────────────
# 🧪 EXERCISE 4 — Model-centric vs data-centric
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("🧪 EXERCISE 4 — MODEL-CENTRIC vs DATA-CENTRIC")
print("=" * 60)

# 1. Try different models on NOISY labels to beat the consensus F1
results = {}

# Baseline RF on noisy (already computed)
results["RF (noisy)"] = f1_noisy

# GradientBoosting on noisy labels
f1_gb = train_eval(df["label_recorded"].values,
                   model=GradientBoostingClassifier(n_estimators=400, max_depth=4,
                                                    learning_rate=0.07, random_state=0))
results["GBM (noisy)"] = f1_gb

# Larger RF on noisy labels
f1_rf_large = train_eval(df["label_recorded"].values,
                         model=RandomForestClassifier(n_estimators=600, max_depth=None,
                                                      min_samples_leaf=2, random_state=0))
results["RF-large (noisy)"] = f1_rf_large

# Consensus (our data-centric target)
results["RF (consensus — data-centric)"] = f1_consensus

for name, score in results.items():
    marker = " ← DATA-CENTRIC BASELINE" if "consensus" in name else ""
    print(f"  {name:42s}: F1 = {score:.3f}{marker}")

best_model_centric = max(v for k, v in results.items() if "consensus" not in k)
gap = f1_consensus - best_model_centric
print(f"\n  Best model-centric (noisy) F1 : {best_model_centric:.3f}")
print(f"  Consensus (data-centric) F1   : {f1_consensus:.3f}")
print(f"  Still behind by               : {gap:+.3f}")

# 2. Did model-swapping beat re-labelling?
# No matter which model we tried (RF, GBM, larger RF), none trained on the noisy
# single-annotator labels matched the F1 of even the simple RF trained on majority-
# vote consensus labels. The ~3-4 point F1 gap is entirely a *label-quality* problem —
# the model cannot "unlearn" systematic label bias (rough-but-OK parts mis-called as
# defect) by being more complex; it simply over-fits the wrong signal.
# This is the core data-centric argument: fix the labels first.


🧪 EXERCISE 4 — MODEL-CENTRIC vs DATA-CENTRIC
  RF (noisy)                                : F1 = 0.504
  GBM (noisy)                               : F1 = 0.509
  RF-large (noisy)                          : F1 = 0.517
  RF (consensus — data-centric)             : F1 = 0.667 ← DATA-CENTRIC BASELINE

  Best model-centric (noisy) F1 : 0.517
  Consensus (data-centric) F1   : 0.667
  Still behind by               : +0.150


#5. Active learning & weak supervision

In [14]:
# -----------------------------------------------------------
# 🔹 5A. ACTIVE LEARNING — LABEL THE MOST UNCERTAIN PARTS NEXT
# -----------------------------------------------------------
# Simulate a small labelled budget: which unlabelled parts should we send to inspection?
rng = np.random.default_rng(0)
labelled = rng.choice(idx, 150, replace=False)
pool = np.setdiff1d(idx, labelled)
m = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=200, random_state=0))
m.fit(X[labelled], df['true_defect'].values[labelled])
pool_proba = m.predict_proba(X[pool])[:, 1]
uncertainty = 1 - np.abs(pool_proba - 0.5) * 2     # 1 = most uncertain (proba near 0.5)
next_to_label = pool[np.argsort(-uncertainty)[:20]]
print('20 most informative parts to label next (indices):')
print(next_to_label[:20])
print('Active learning spends the labelling budget where the model is least sure.')

20 most informative parts to label next (indices):
[1027  737 1721 1760 2131 1723 1862  960  626 1276  891   84 1998  435
 1333  398  855 1535  196 1551]
Active learning spends the labelling budget where the model is least sure.


#### 🧪 EXERCISE 5 — Weak supervision: rules as labels
Sometimes you can label *programmatically*. Write 2–3 **labelling functions** (heuristics) over the feature columns, e.g. `porosity_pct > 6 -> defect`, `wall_thickness_mm < 4.5 -> defect`, `surface_ra_um > 5 -> defect`.
1. Apply your rules and combine them (e.g. majority / 'any rule fires') into a weak label.
2. Compare the weak label's accuracy vs `true_defect` to a single inspector.
3. In a comment, note where rules beat humans (consistent, scalable) and where they fail (miss subtle cases).

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 🧪 EXERCISE 5 — Weak supervision: rules as labels
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("🧪 EXERCISE 5 — WEAK SUPERVISION")
print("=" * 60)

# 1. Labelling functions (heuristics from domain knowledge)
def lf_high_porosity(row):
    """High porosity → defect. Threshold from EDA (90th pct ≈ 5.8)."""
    return 1 if row["porosity_pct"] > 5.8 else -1      # -1 = abstain

def lf_thin_wall(row):
    """Thin wall casting is structurally weak → defect."""
    return 1 if row["wall_thickness_mm"] < 4.5 else -1

def lf_rough_surface(row):
    """Very rough surface → defect (surface_ra_um high)."""
    return 1 if row["surface_ra_um"] > 5.0 else -1

def lf_slow_fill(row):
    """Slow fill time → more air entrapment → defect."""
    return 1 if row["fill_time_s"] > 3.2 else -1

def lf_high_temp(row):
    """Very high melt temp can cause excess porosity → defect."""
    return 1 if row["melt_temp_c"] > 735 else -1

lf_fns = [lf_high_porosity, lf_thin_wall, lf_rough_surface,
          lf_slow_fill, lf_high_temp]
lf_names = ["high_porosity","thin_wall","rough_surface","slow_fill","high_temp"]

# Apply all LFs; -1 = abstain, 1 = defect
L = np.array([[fn(row) for fn in lf_fns] for _, row in df.iterrows()])

# Coverage and accuracy per LF
print("  Labelling Function Analysis:")
print(f"  {'LF':<16} {'Coverage':>10} {'Accuracy':>10} {'Precision':>10}")
print("  " + "-" * 50)
for i, name in enumerate(lf_names):
    votes_i   = L[:, i]
    covered   = (votes_i != -1)
    coverage  = covered.mean()
    if covered.sum() > 0:
        accuracy  = (votes_i[covered] == df["true_defect"].values[covered]).mean()
        # Among rows it labels defect, how many really are defect?
        labeled_defect = (votes_i == 1)
        precision = df["true_defect"].values[labeled_defect].mean() if labeled_defect.sum() > 0 else 0
    else:
        accuracy = precision = 0
    print(f"  {name:<16} {coverage:>10.1%} {accuracy:>10.1%} {precision:>10.1%}")

# Combine: majority vote among firing rules (abstains excluded)
def combine_weak_labels(L, strategy="any"):
    """
    strategy:
      'any'      — defect if ANY rule fires
      'majority' — defect if majority of voting rules say defect
    """
    n = L.shape[0]
    labels = np.zeros(n, dtype=int)
    for i in range(n):
        votes = L[i][L[i] != -1]   # drop abstains
        if len(votes) == 0:
            labels[i] = 0           # default: ok (no rule fired)
        else:
            if strategy == "any":
                labels[i] = int(votes.sum() >= 1)
            else:
                labels[i] = int(votes.mean() >= 0.5)
    return labels

weak_any      = combine_weak_labels(L, "any")
weak_majority = combine_weak_labels(L, "majority")

# 2. Compare weak labels vs single inspector and consensus vs truth
acc_insp_A = (df["inspector_A"] == df["true_defect"]).mean()
acc_insp_C = (df["inspector_C"] == df["true_defect"]).mean()   # least reliable
acc_weak_any  = (weak_any      == df["true_defect"]).mean()
acc_weak_maj  = (weak_majority == df["true_defect"]).mean()

print("\n  Accuracy vs Ground Truth:")
print(f"  Inspector A (best human)    : {acc_insp_A:.3f}")
print(f"  Inspector C (worst human)   : {acc_insp_C:.3f}")
print(f"  Weak labels (any rule fires): {acc_weak_any:.3f}")
print(f"  Weak labels (majority vote) : {acc_weak_maj:.3f}")

# F1 scores (more informative for imbalanced data)
f1_weak_any = f1_score(df["true_defect"], weak_any, zero_division=0)
f1_weak_maj = f1_score(df["true_defect"], weak_majority, zero_division=0)
f1_insp_A   = f1_score(df["true_defect"], df["inspector_A"])
f1_insp_C   = f1_score(df["true_defect"], df["inspector_C"])
print(f"\n  F1 Scores vs Ground Truth:")
print(f"  Inspector A                 : {f1_insp_A:.3f}")
print(f"  Inspector C                 : {f1_insp_C:.3f}")
print(f"  Weak labels (any)           : {f1_weak_any:.3f}")
print(f"  Weak labels (majority)      : {f1_weak_maj:.3f}")

# 3. Rules vs humans: where rules win & where they fail
# WHERE RULES BEAT HUMANS:
#   - Consistency: rules never have a "bad day" — they apply the same threshold
#     to every row, eliminating the random drift you see with Inspector C.
#   - Scalability: once written, rules label 2400 rows in milliseconds vs hours
#     of human inspection time. They're trivially parallelisable.
#   - Auditability: rule triggers are explicit ("porosity > 5.8") — easy to
#     explain to a quality-control manager.
# WHERE RULES FAIL:
#   - Subtle combinations: a part might be barely-ok on every individual
#     dimension but a combination (moderate porosity + slightly thin wall +
#     slow fill) makes it fail. Rules miss non-linear interactions.
#   - Calibration: our thresholds (porosity > 5.8) were hand-set from EDA;
#     a human inspector integrates visual cues and experience that are hard
#     to quantify as a threshold.
#   - Abstentions: LFs abstain on the majority of rows (low coverage), leaving
#     many castings unlabelled — a model trained on this data sees limited
#     supervision.
# Best practice: combine rules (weak supervision) with human labels (active
# learning) using a framework like Snorkel to learn a weighted label model.



🧪 EXERCISE 5 — WEAK SUPERVISION
  Labelling Function Analysis:
  LF                 Coverage   Accuracy  Precision
  --------------------------------------------------
  high_porosity          3.5%      96.4%      96.4%
  thin_wall              2.7%      49.2%      49.2%
  rough_surface          2.2%      46.2%      46.2%
  slow_fill              5.1%      37.4%      37.4%
  high_temp              4.7%      50.4%      50.4%

  Accuracy vs Ground Truth:
  Inspector A (best human)    : 0.955
  Inspector C (worst human)   : 0.888
  Weak labels (any rule fires): 0.752
  Weak labels (majority vote) : 0.752

  F1 Scores vs Ground Truth:
  Inspector A                 : 0.916
  Inspector C                 : 0.802
  Weak labels (any)           : 0.426
  Weak labels (majority)      : 0.426


#📘 Summary

| Technique | What it buys you |
| --------- | ---------------- |
| Cohen's kappa | quantifies annotator (dis)agreement |
| Consensus labels | cancels random annotator error |
| Error detection | finds likely-mislabelled rows to fix |
| Clean-then-train | better labels lift the model (no model change) |
| Active learning | spend the labelling budget where it matters |
| Weak supervision | label at scale with rules |

**Core lesson:** in the data-centric mindset, *the labels are part of the model*. Measuring and improving label quality is often the highest-leverage thing you can do — frequently beating a fancier algorithm.

**Next — U20:** once a model is trained on good data, is it **fair** and **explainable**?